In [1]:
from langchain_core.prompts import ChatPromptTemplate

In [2]:
CONST_RULES_QUESTIONS_CONTEXT_RETRIEVAL = """
You are an expert AI Data Engineer specializing in Information Retrieval (RAG) for a World of Warcraft EN → PL translation system.

Your task is to analyze the provided quest text and generate between 2 and 3 distinct search queries (questions) that will be sent to a vector database (embedding-based retrieval). 

These queries must retrieve the underlying lore, entity backgrounds, and world-building context that a translator needs to accurately capture the tone, constraints, and meaning of the text.

CRITICAL RAG OPTIMIZATION RULES:
1. DO NOT ask "How are X, Y, and Z connected?" or list multiple anchors in one question. Vector embeddings fail on over-constrained, cluttered queries.
2. Isolate concepts. Each question must target ONE specific entity, phenomenon, or faction relationship to fetch its objective background lore.
3. Write queries as clean, direct, factual questions. Avoid phrasing like "What does character X mean when they say..." or "Why is X reacting this way...". Instead, search for the underlying lore fact (e.g., "What is the nature of the Lightbloom phenomenon?").
4. Never use prohibited phrases: "in this quest", "in this scene", "Warcraft lore", "in the lore", "mentioned above".
5. All outputs must strictly use double quotes ("") for both aspect and question. No single quotes allowed.

EVALUATION CRITERIA FOR QUEUSTIONS:
- Would this question match a heading or a paragraph in a WoW lore encyclopedia (e.g., Wowpedia)?
- Does it avoid meta-commentary about the quest dialogue?

EXAMPLES

Example 1
Quest text:
Title: Ashes Over Theramore
The ruins of Theramore still smolder, and Jaina Proudmoore has refused to leave the shattered tower. Those who knew her say grief has hardened into something colder. If she is pushed too far, restraint may no longer hold her back.

BAD (Too cluttered, useless for vector search):
aspect="Jaina's grief" question="How are Jaina Proudmoore, Theramore's ruins, and grief hardened into cold restraint connected in this scene?"

GOOD (Clean, targets factual lore background):
aspect="Jaina Proudmoore post-Theramore behavior" question="What happened to Jaina Proudmoore during the destruction of Theramore and how did it change her personality?"
aspect="Destruction of Theramore consequences" question="What are the political and magical consequences of the destruction of Theramore?"

Example 2
Quest text:
Title: Silk and Shadows
The nerubian vizier insists the sealed lower tunnels of Azj-Kahet are empty, yet the workers hear whispers beneath the webbing. Black blood stains the old stones, and several scouts now serve a voice they refuse to name.

BAD (Asks about local dialogue instead of lore):
aspect="Nerubian whispers" question="What does the nerubian vizier hide and what is the unnamed voice that the scouts serve?"

GOOD (Targets the underlying entity/phenomenon):
aspect="Azj-Kahet lower tunnels lore" question="What entities or dark forces are trapped beneath the sealed lower tunnels of Azj-Kahet?"
aspect="Black blood in Nerubian lore" question="What is the significance of black blood and unnamed whispers in Nerubian culture?"

Strictly follow this output format (copy it exactly, replace only the text inside quotes, use EXACTLY ONE newline between items, NO markdown blocks):

aspect="ENTITY_NAME" question="CLEAN_QUESTION_1"
aspect="ENTITY_NAME" question="CLEAN_QUESTION_2"
aspect="ENTITY_NAME" question="CLEAN_QUESTION_3"
"""

In [3]:
hash = "eJztPdtyGzey7/wKeLYcJ1UiObyKoiSqfMnF5ziO1vYeJ5VKscCZ5hARBpgAGFLMqVO1n7CfsN+yn7JfcqqBuZEc0lTieL1r5SEWgUaj0Wg0uhsNzEXIliTgVOtLbwmBkao5k+HaIyy89PCvp1IYEMabNC4qoEImignjwDQz8DqdeZOvlIzJW6oCReeGvGU37KIdsmXWFEEDh81C76/r7lb+nMZJ08imoMuykubkxKumBeBM3HhkoWB+6f0pXjUXQENv8l9pnBAjiaBLFlHDpLho03c210BVsHguktSUKFyha15hR7yyXGtmQ3BsiVf576aBW+NNtlrkldwoEq+aCVUaVFOmBnskIVOXHjfKI5yK6NID4U0uDJ1xIAFwrhMaMCzveTlKJuZyJm9JSNWNA/wlBW1m8tYj2qw54ISqEFQzazzuJbfnKxaaxbjbhTj7k3T91gDi83PsD0c1uTAq74PO5BIsW0Fh/YIEEmkRl1636AVH26ScRWIcgDCgzudSmKZmv8K40x08dD9XwKKFGc8kD88TGoZI0CC5xW4RYY4sZDrhdD22I8pI7Pj+w/NsLIHknCYaxvkf51uD9A9ibCq5OgyA3D5fgjIsoDwbVszCkENOdlPZkSDxGX2VYWSM8yYXNJOs9ordsPZXNEBJ9IhhBjt9CalRlHuTCxZHhHJTFpGQGtqcM46cx64uvV63Wmp7vfQ6I4+EEEik6dKjei0Cj1RbcEldHae/rj2iVXDptVlMI9DtrLNpr9tKRHRFYQhD3yMl6vYExf6ijaMq/nkPbOtaZr3Av2dcypi8kDJmItrqKf/HLPB/CoXSyp8OZAKXnpKrOvnjMDfnMxrcREqmIkRpkWpsFBU6oQqEOfcmrw1VxiE2oVUL1Xn6Tq1gLWgxT/nvSfaHY4oJ3zdVX4rwo6PpBSyBl1SN/Oao90f085QaiKRa72XAl0tQWopo+lbKUBd8yIuJK55s/v7DZuo2AcVABFDS2+md+Gf+H9HbK1hRhYPJu/pOAJHzManuLgmnTHCm7bad8skFZ5vqSMiZAnqzpaRw/2OBFFUlVKd9hv0a7WML92mfzvCw9jGLNJ61n4vlNGY6mNqNi0kRU3WD6qjdGSa3zb3VV53TYX8eltpq6LUn//h7rjM2ZcdqGm0ovwH1sHuqp685SxJQpRhVIR5pUtRvMosZiJFhhP5Mb42xf/4SNFMRyDiuKPY92AjThAqSg5NfUsqZWRNE29rqaqZocAPGm/yYDWkPzvpGP22r0IoW52zSuBeOA8LxRgGtaJgtxme170UwHK73KRYO471QvHeheCKl2SsTrvK9iIRF9T4lwiL8jxeIhFMDbsrRogyBz6ezqd+ZzqQ0m8LxTtCrYDg/g+A3CsrXCugS9opKXv1ehCVD9j7FJUP5mwTmop3yScN56mia1AqPHaJMDShjvEm3d1HH1W+lgPWf3qwT0FM5nwaSiZKjUVXQojpJ6/o1kmYL90pa/xhJ+1ry0AlTP7lt5r+ugu6oNzor5aW/6TiRvn/3QerqIPWHHORrxpegymGWv6/o/HTkh3sHWvxj5/4PMISvFSyZTAtLOBctu5j1Qipr/FaFztbYGM+kplyKqJkomLNbUqfEXtaxvTOoc8MHh9g+OM4N7wwck0/PwhktmTyw2udH63n9tE8NvV5QlcR0Nf3cjuyLQo7yCpJXbPKhqm9KVuWNdtf5Cfkk2fsVZWqmAH6F6Vd0KSuOQ1lDsppjGLzTqo7Tn9E4Of80+f3MxpUCCKfPQLBfoaIYiypSVB3D8d1mdZvY+1dZL+G2CDTdq6t8ft8oFmPAb2oWMC3jgMUk5/XELIBU6o+Z6T1td6f7U15gT1OFG+n0acoNW9oDkoL5WR2p1h3D+Jp2B9dYOztqaNvo8cZRiZGSG5aUhytb8WYhBZyT3UOH7MAlg55zSY1bpf/poXm0uGri6TPVdhHcMXGRWywoD8FsZ94mIxiH8eMlZRwnxfnQgRRIRWlTYPR8XOVNVlIn/sM61gx/h9NXT9zVWSc4Df1Npw1ZRH5LIP1INgWGLffw6EsRbnDI/v4w/Nml6mo474w675M531+PSRbwRtHLItRjJ4j34en78PR9ePo+PH0fnr4PT9+Hp+/D0/fh6fvw9McXni6TDalitMnpDDiHcLa2mXpGBjbnjaFNX3ilgUvzs38oiYwoEwyRI5g5WIE2Moo4BAsIbmxKXtZ2u9hhmqXG4FqqcXU9YtYYhyqabDnLgZ2VPQmEi26emlgd0yRL8tQX7UV3cyUUFGKhN7mwnNmptKUemUtVNyZccQiwyXRr+aPermDj6J42O8TIQIN1wpudinv+p+9mP2PxrlIyMhBpPMOkxE4uyjvjsEmYJYoNzZDtIYdp6VZpeQY6UCypiYhUiOkeJqaC4+7U9KrUZE7Xfkp6hykp0oruSkW/SsVTGSccDrOkf5iQEsXdaRlUaXkpzSExGRwmw7a+OwXDKgXXSkYKtD7IjuFhOio47k7N6QY11ASLabCgIjrEl9N30INYSIbl7hSNqhR9eWtACcqnaEMcIGl0mKQcjQ3D1tCU7fRWHxe6Z1vPZcninKGCRRVZVTU1OmPRnTQukslXknO5stFloyjjRM7tj7A4YVgxHnI2h1bjop2U4Y7XCdoXK2YWvynsUti8b2yvc0sGhESmBkkoj5RKyNcgDBPACdWaaQPhPiQVg7qIeglYbca8yvji9FsptFFSM7O+cpHTSwiZsQF1BcjSm8uORxRwNKxcL5tWootTVvCQzxMaAQklaCKkIXDL7HFlPbQNXmmMKe3li5EEMKG2tPzeKQAb+r1OUWci8GYBxSQTDRBr7G1Bl0BmAKIiCnO8HoHSoWVqFi3y3KA1bBaggKwsEk50qoCvyZyJkFChV6C0k5yLBFUjNJvPHWCqgcRrEtOIBdgh4zyNmaAGbBcJNQtCRVggnm9Jai6P7+RDsbNsbxHZ+H+QqetBQQBsCUS6iN+mvP+b+Gb34b378N5HJBz34b17obgP7/17CcR9eO/jCe9VrBPKtcxNlG3bZL+Z+51Ci2rq4kqxlGL6lIoQ4x2K3t3YddhIiY2U2PaZvIfabLoC9xHMTzOCmc+/O4JHd7I05RWETEFgNqX6++ti5N9fe5PvrwtBOtY5qgZ6aiI2pWukgQQKqEkVaEI5J4GMwXlC6Pe0yJsF05XMrFarRQwWMU1W1jNyDpOIctd6JRUPyYpqwl34ksSFc/1OurOg0GZ0x1Ib8slFaCZ/SaQgNAggMSdEWyfdSJK53tahCjgLbogUpJSYr6XWLPkwGR2ur8j+v0jpoMNwWF0T2TZRZcUvQVNRBd7k8z/b3N9s9P/4+wswj/SOe0jWMiVzzGRsXbRDGzdxCjVnhVOqQSDjhIq1BUfuPFbUSJXPh2VquBXYwUQtTdfe5GKWY/vH3zVdY8bIbEK2QyrWT9YLmfLQzjjOB1Zm3jE6uBrgplUGl/G+6MFej1gibhw2JfEVhAAxqGLNuDpLRFE3cYW4kKqD2VJCjzkHxej0LROhSoWoInVVpFI1+VZicMCFNygLkcmPFJA5jRlnVLnwkV0TjIe6Rd4uqCGhtJMR0xt0w+2awwV1dQf+1MwKRjmKhewWKayJghgDFbHtaSFjOCFrMK6Ou1iAW7+pmtMAqlOJe9wMVzcudLremL5C2L7J5nsGaynCbW6+phz5w7i1dr6hSyjNk6LukSauZrJTZBnrlAwV5IAVUFw3fgZzECGou2/9xVXkHMW+/X4H0FLJNJlRa41FhJLj4nLfrMPfYqRUYmwWwxGxOAuHdB697ndGWZW1H2T6iHMCIlCSBgsiJJmnCufuwUe5yB8Lu07dBiU1xv2oNvoB+XPKghu+flAj25Uju5BRLqMUSCUtrq6+6Q727MLY4V/G+Cy6nUxseBSUfoArd41aI7BZoGUQspy/B3mA8aPf0R4rcDsNhhsR+9X2uO3+rTld7yZ9OzE9eS8i+u2amAUVN7r1+9TqWyBxqq0mzFSlHRbufMHNYZ24pQzRPgYxk2m0mF5TXdq2lQpiKybbJVU9SEJEf0DBPBdzwNOD6bc0wmOE36ALcxQkR7FPxewAOkqpJjPkPxypCK/BUL5iKFW/QxkWWI5QiAWs4y0aRlZrbzgkLBTYYspEM6IxTAMmIKaGBeXc5TAkgyEVmIkCypuGYWlqdAACbGcbClwqJRWmEVBlNCoYXBf5EpizKEXF4DgI8QzCJQtBZgs8XjXZXNEYAinmLLp89L+eVoE39hbGJHrcbq9Wq9ZapiadQVPIQMobBq1Axm2Lqv3y6x9uXrJe/PXL/hVNjcTeLzve/z1y6DWoJQvg0stQFMkO7g2dYd9PbrdenSpJbK4UxaBz0cgpm3FvaFvtaRRIoe0NAkfAQq6aiWJLGqybQhpLTGdvY7kExe1Krq9HFYcEbSRwHAaeTuf0Bp5koC8kDYmtLtOSDjWV0thHmA6DZWz2MBLyJp1BNePlYD8ZZ15axpAFC0ObQXIE8HSaX9XIOyWxlWJ8ogkCQ/C0QOKJLc5Dq7KK92OsPs1VI39W7BZylf1cSXWj2ym+p6XBGFyI7Qxhe2upnwgpE3CWOFURmEtvOuMU77BcuxbkWnIW2FPG4xk2nToJwMiC++s4rjGRFhk+Op3FzKZEuPKLtsN0PMaQ6ZjZvWAD4TNXXOLblYmNvKi20xSVXZNlptBJ7hqfWCWHOvlxuASBboIiCkOB9t006g4kVwsWWI9lTbSRSeJOZjU4L98oJtGIqlWkJE6DBWrfbJ/c2RtwK2YH9+GZ1BpiaeAD2IeF17QRAMHjYTe85oyD1hCekFnqHCa00bRE12rJArw7VWNaMHvabBYKoMqaGURMWL/YOX3zO7Iim8OC5jXCshhnB90zag+pmSIGlGJGqnULDReqgAi5Kux0JqJ6i+W5WOL5B2bGZUbZu4yMf73DRedolyCtxZabR43fYbrjiD/j5twGtsIM5yNNAqkSDWSB1gvmBWi6pBGEaMrMbKQW1w/lUrTIY22tXLilMRNubawwFKNPCCUGRKhcpskSIjAUE1NIpOQK76gmSqZG46qzhj5GCFA0FkDwfl3rs8icb5vMX8s8pFIspWzKrUxW/M769Iujnb4P55G9UTLFBxApRv8ekO8E8lORKKUqfHAXq7piRT42i4fd01d06wjEseOxWTzCqsluWcW6LqNrKAU2Emp1Y8ClhruwcYNEzFgVgk5fsSWoFRNhSWBWRcqqyW7ZDvPQC0kFaCMFoD7CFBLrHCBLMbdF8pDgVWc7ICd5To/hurIucBbGFZH+A5zf3SFsi7TTu3YA2uqpVKeU8zVmzYQo3ImSxqWkYdIqQeGYS4U3sQuFTHSKJg/GriIb39oMUru4Gnr722ExzOahqGMso+zKEWtEwIzl2oNNat8Zs95II6zLB7Txa1mTZ/OpXGCuHM+9hEgaZnOwSzeqcni2UX/U8xf1bY8+qP5U5uAZS7iMabDeeXWkqLnTsyNFqw/D6ScfhtNPpFnkbIbuYOCP7sjma7qEcMrE9LFeFBy2hYQJgoXHMLfa4F6Sa953mT6x2bMn7j2K6ZONBykQgGQAblcgDuDY117qGt9Pw/0rRv/hrxh9ym+s/KseMfrkVEhpRWvgc8u0/I+j2HrgIfJPma0fxXtBn7IC+SheyfrEBf81wA1OwRuVmtL6zkqJKz3KGKm2uOfyfm/+21Sb6X+LSli34o5jJbGVd/Tji4Z7k4flUUmQWzcq669G2uDMxiWorTvdGPUS0JxxGdyck5gJNxFjMvCT25p3zbbztQMpSCyaDs3OW2ZvMVsSE17zz/KMp/mh8s4pc1VeSqBdsRnUZc93/EPJsN3uMcmweafZYGxWLB4DN2sqrkZ+pz8cFQfAW58aIe5bI4Xo9f3a7FmyG091s9rptvxWp+Jh49xmhZPqLxve/bzrd4dNv9vs+F9kOUFhCGGLEHKB8sSyV8TukmK7fTt2zzXXTL6Ks8AMunoVdyVX7kNIlYPWHJm9Qltzrpq1seeqdhVdjoan/Z3TlslbB1d7BVizSDG4a7fhrOxUt/f1+vbZk50rvg+aTdJ4Cavra8JZzAxRkEhlGk9psMCzLBYDft2oO/R7nVFn1OkMzlwdgduEqfWYdAdnXd/3G68gTNGyzsvnlGto2ERnFrgA4Jj8iAkE//zr34wMfmo8vf6L7YCkmkYwJn6rMxwSjbkToW68AhzzRnVv5BfV1woSJQPQWiqyZPg9LUyQDDFpKxVmTLr9wajd8e1/jVeAIJjmhB9UIr2R77f7nbN+z++T2dqAblxLbf7517/BbYKHSkwEPA0hA++e9nu9Lfg3ENurKISqKI1BmAx2MOqfboF+w6IFaDw9S6iwVISQoLbq9JDABn4MRWg8kXLfsyLzVLg4ezYUvz3w/cZf8PSIJXghI1VVNH67W9YmG+NwNHX800HOCj8j6stb80LKRG/04TdepHSL6b5/1u4MWtgy5z0CxRBLtc7BhqOzzlm/Pej2u6NR0cdLexkdw/jZVEFInlIVSdxkFAM9Jn7jDfamE+RgBoXb/DaY6598TpdRK//1RaUDB6/kijChUa1hSroxECdGj0mn0Ww6YW+8wddlA55aBpYzYsfsZJ98/vAk1icB5VyfmGyav2h0fOz2ISFdf9Dqdc6I/a9DmkYayhuk32kN+g8JIaNBq3s2yKtzORnbFPKZvG2QXrflnyHkcNAanQ13IJ+7j5I1SLfXGo4QsD9qDbu9HcDXJp01SLeTQ/VbA39UC9W2vTdI56x1NrCwfuvs1K+HfYJ9d/putKQ7ap0Odvt+HIds2SCdXms0zMD6OVi3BHsqsVsybJ1aKAQfjXagXDpjg5BBa3Rq4botf9DZ6bS6zxRTSl7nkdls+QRWP9lE7xtYk1W2i09BjBNbN2bhgurFeDjw+71B038QUCEF7oTuRJnFoA2Nky3FZyvxBXUrMSwkw+Fpf9Aftsgre2CNcounWEaxKAKF59QQ0FTDmOAZfBPP0hvEkl15PCdT8/bbgFl+krMUhNQCbwnjfbzJKzCKAY7TnlF7F7TyWM3WnpCNt4VbcyuKdk5lMx/9Ck/awstsEN7k97XHncXbfRooG11ArUmZXxOKV/jpPQM2w91t3QXE9if/hFQx5c0SQ/YYz3b5jkmcQMAoH2cfxmKVS301Vfn3sxiaoXQyJpWLbht+fPaZrfHm57TchcSyhx0w9zktZ1zvfGUrKy5vFB3oN/cW9nWY1Rc9bf4+qgurK/SUmql1Onb7cACEGvcorjfZKjiqF2vt4alMUmc8boK5xV01J2uLN69k7Saeob1oc/IKqSEbRTr7u5CxLWhv8o0tIEEhLO8WlGs01qcp7mrTLzHr7H9cvuj2QC0csXCkAjepLz9+IqfapLMaQbGVxFW6CXQ/jsL8VYqn99nU7enAwWQTlPWzW1Y7Z3kOHf74f7E/fyY="

In [ ]:
import sys
from pathlib import Path

for katalog in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.cwd() / "python-etl", Path.cwd().parent / "python-etl"]:
    if (katalog / "scraper_wiki_main.py").exists():
        sys.path.insert(0, str(katalog))
        break

from moduly.utils import hash_do_wsad_json

wsad_json = hash_do_wsad_json(hash)
#print(wsad_json)


In [5]:
import json

data = json.loads(wsad_json)
misje = data["Misje_EN"]
misje_tekst = json.dumps(misje, ensure_ascii=False, indent=2)
misje_tekst

'{\n  "Podsumowanie_EN": {\n    "Tytuł": "Lightbloom Looming"\n  },\n  "Cele_EN": {\n    "Główny": {\n      "1": "Follow the trail of the displaced wildlife."\n    },\n    "Podrzędny": {\n      "1": "Speak with Orweyna",\n      "2": "Trail followed out of Fairbreeze",\n      "3": "Sentinel assisted",\n      "4": "Trail followed",\n      "5": "Lightbloom Monstrosity slain",\n      "6": "Trail followed to end"\n    }\n  },\n  "Treść_EN": {\n    "1": "The wildlife seems to have been displaced from the south. It is there we will surely find answers.",\n    "2": "Come--I will use my magic to illuminate the path and we will follow the trail."\n  },\n  "Postęp_EN": {},\n  "Zakończenie_EN": {\n    "1": "These creatures all come from here. This Lightbloom... this is where the song of the world was leading me."\n  },\n  "Nagrody_EN": {\n    "1": "You will receive one of:",\n    "2": "You will also receive:"\n  }\n}'

In [6]:
import sys # aby importy z dołu działały
sys.path.append(r"C:\____Moje-MOJE\MyProjects_4Fun\projects\World of Warcraft\python-etl\moduly")

In [7]:
from moduly.ai_klasy import QuestLoreResult

In [8]:
prompt_questions_lore = ChatPromptTemplate.from_messages(
    [
        ("system", CONST_RULES_QUESTIONS_CONTEXT_RETRIEVAL),
        ("human", """
            {misje_tekst}
        """)
    ]
)

In [9]:
def lore_question(llm, mission) -> QuestLoreResult:

    structured_model = prompt_questions_lore | llm.with_structured_output(
        QuestLoreResult,
        strict=False,
        include_raw=True
    )

    result = structured_model.invoke(
        {
            "misje_tekst": mission
        }
    )

    return result

In [10]:
from moduly.ai_modele import llm_lore

In [11]:
llm = llm_lore()

In [ ]:
res = lore_question(llm, misje_tekst)

In [13]:
for question in res["parsed"].questions:
    print(question)

aspect='Lightbloom' question='What is Lightbloom and what kind of natural or magical phenomenon is it?'
aspect='Orweyna' question='Who is Orweyna and what is her role or background?'
aspect='Fairbreeze Village' question='What is Fairbreeze Village and where is it located?'
